# Make a song

This makes a full song from your words. It runs on Kaggle's computer, not yours, and it is free.

## Set two things first

Look at the panel on the right side of the screen.

1. Find **Accelerator** and set it to **GPU T4 x2**
2. Find **Internet** and turn it **On**

If you skip these it will not work.

## Then

Click **Run All** at the top. The first time takes about 10 minutes because it has to fetch
the music program. After that, songs take a few minutes each.

When it is done there is a play button near the bottom, and the song file shows up in the
**Output** panel on the right so you can download it.

## To write your own song

Scroll to the box that says **YOUR SONG**. Change the words in there. That is the only box
you ever need to touch. Then click Run All again.

Kaggle gives you 30 hours of this a week, free. A song is a few minutes, so that is a lot
of songs.


## Step 1: check the computer

Makes sure Kaggle gave you a graphics card. If it complains, set **Accelerator** to **GPU T4 x2** in the right panel and start over.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| GPUs", torch.cuda.device_count())
assert torch.cuda.is_available(), "No GPU. Set Accelerator to 'GPU T4 x2' and restart the session."


## Step 2: fetch the music program

Takes a couple of minutes. Nothing to do here, just let it run.

In [ ]:
import os, subprocess, sys

COMFY = "/kaggle/temp/ComfyUI"
if not os.path.exists(COMFY):
    !git clone --depth 1 --branch v0.33.1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}

!pip install -q -r {COMFY}/requirements.txt 2>&1 | tail -5
print("ComfyUI ready at", COMFY)


## Step 3: fetch the music brain

About 12 GB. This is the part that actually knows how to write and sing music. Give it a few
minutes.

It re-fetches every time you start a fresh session. That is normal.

You do **not** need a Hugging Face account for this. If you happen to have saved a token in
Kaggle's Secrets under the name `huggingface`, it will use it. If not, it carries on fine.

In [ ]:
from huggingface_hub import hf_hub_download
import os, shutil

# Optional: if you saved a Hugging Face token in Kaggle Secrets under the name
# "huggingface", this uses it. If you did not, it just carries on without one.
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret("huggingface"))
    print("Signed in to Hugging Face.")
except Exception:
    print("No Hugging Face token found. Not needed for this, carrying on.")

COMFY = "/kaggle/temp/ComfyUI"
REPO = "Comfy-Org/MiniMax-Music-3"
FILES = [
    ("diffusion_models/minimax_music3_dit_int8_convrot.safetensors",           "diffusion_models"),
    ("text_encoders/minimax_music3_text_encoder_pruned_int8_convrot.safetensors", "text_encoders"),
    ("vae/minimax_music3_dav.safetensors",                                     "vae"),
]

for remote, folder in FILES:
    dest_dir = f"{COMFY}/models/{folder}"
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, os.path.basename(remote))
    if os.path.exists(dest):
        print("already have", os.path.basename(remote)); continue
    print("downloading", remote, "...")
    p = hf_hub_download(repo_id=REPO, filename=remote, cache_dir="/kaggle/temp/hf")
    os.symlink(p, dest)
    print("  ->", dest, f"{os.path.getsize(p)/1e9:.2f} GB")

print("\nAll weights in place.")


---
# YOUR SONG
---

**This is the only box you ever change.**

**CAPTION** is how you describe the sound. Say the style, the speed, the instruments, and what
the singer sounds like. The more you tell it, the closer you get. Keep the three headings.

**LYRICS** are your words. Mark the parts with square brackets so it knows the shape of the
song: `[Intro]` `[Verse]` `[Chorus]` `[Bridge]` `[Instrumental]` `[Outro]`.
For an instrumental, delete the lyrics and say "instrumental, no vocals" in the caption.

**DURATION** is how long the song is, in seconds. 300 is the longest (five minutes).
Try 60 first so you can hear how it sounds without a long wait.

**SEED** is which version you get. Same words plus a different number gives you a completely
different take on the same song. Change it when you want another go at it.

**STEPS** is how much care it puts in. 30 is normal. 50 sounds a bit better and takes longer.

**CFG** is how closely it sticks to your description. 1.7 is normal. Higher follows your words
harder, lower lets it wander.

In [ ]:
CAPTION = """Global Metadata: Gritty blues rock, 96 BPM, E minor. Raw and driving, a dusty roadhouse stomp that builds through the chorus.

Vocal Details: Gravelly male lead, weathered and full-throated, sung slightly ahead of the beat with real bite. Rough gang-vocal answers on the chorus.

Arrangement: Dirty overdriven electric guitar, thumping kick and snare, walking bass, tambourine, a wailing slide lead over the chorus."""

LYRICS = """[Intro]

[Verse]
Steel in my hands and the sun going down
Sparks on the deck of a nameless town
I built every beam that you drive on now
Nobody asks me how

[Chorus]
So I sing it loud, I sing it free
Every road out here belongs to me
Ain't got a dollar but I got my say
And I'm singing it anyway

[Verse]
Hands full of callus and a voice full of smoke
Every song I ever wrote came from something broke
They can keep the ceiling, I'll keep the sky
I ain't done yet, and here's why

[Chorus]
So I sing it loud, I sing it free
Every road out here belongs to me
Ain't got a dollar but I got my say
And I'm singing it anyway

[Outro]
"""

DURATION = 60      # seconds, up to 300
SEED     = 222
STEPS    = 30      # diffusion steps. 30 is the shipped recipe; try 50 for more polish
CFG      = 1.7


## Step 4: start it up

Nothing to do. It is warming up in the background.

In [ ]:
import subprocess, time, urllib.request, os, sys

COMFY = "/kaggle/temp/ComfyUI"
LOG = "/kaggle/temp/comfy.log"

def comfy_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3)
        return True
    except Exception:
        return False

if not comfy_up():
    logf = open(LOG, "wb")
    subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
                      "--disable-auto-launch"],
                     cwd=COMFY, stdout=logf, stderr=subprocess.STDOUT)
    for i in range(180):
        if comfy_up():
            print(f"ComfyUI up after {i}s"); break
        time.sleep(1)
    else:
        print("ComfyUI did not start. Last 40 log lines:")
        print(open(LOG, errors="ignore").read()[-4000:])
        raise SystemExit(1)
else:
    print("ComfyUI already running")


## Step 5: write the song

This is the part that takes a few minutes. You will see a timer counting up.

The first long stretch is it working out the tune and the singing. It looks stuck. It is not. Let it go.

In [ ]:
import json, urllib.request, time, uuid, os

S = "http://127.0.0.1:8188"

graph = {
    "1": {"class_type": "UNETLoader",
          "inputs": {"unet_name": "minimax_music3_dit_int8_convrot.safetensors",
                     "weight_dtype": "default"}},
    "2": {"class_type": "CLIPLoader",
          "inputs": {"clip_name": "minimax_music3_text_encoder_pruned_int8_convrot.safetensors",
                     "type": "minimax", "device": "default"}},
    "3": {"class_type": "VAELoader",
          "inputs": {"vae_name": "minimax_music3_dav.safetensors"}},
    "4": {"class_type": "MiniMaxMusic3TextEncode",
          "inputs": {"clip": ["2", 0], "caption": CAPTION, "lyrics": LYRICS,
                     "seed": SEED, "max_duration": float(DURATION),
                     "cfg_scale": CFG, "top_k": 50}},
    "5": {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["4", 0]}},
    "6": {"class_type": "EmptyMiniMaxMusic3LatentAudio",
          "inputs": {"seconds": ["4", 1], "batch_size": 1}},
    "7": {"class_type": "KSampler",
          "inputs": {"model": ["1", 0], "positive": ["4", 0], "negative": ["5", 0],
                     "latent_image": ["6", 0], "seed": SEED, "steps": STEPS, "cfg": CFG,
                     "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0}},
    # 16 GB holds this comfortably, so plain decode: no tiling, no seam risk
    "8": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}},
    "9": {"class_type": "SaveAudioAdvanced",
          "inputs": {"audio": ["8", 0], "filename_prefix": "minimax_song",
                     "format": "flac"}},
}

t0 = time.time()
req = json.dumps({"prompt": graph, "client_id": str(uuid.uuid4())}).encode()
pid = json.load(urllib.request.urlopen(S + "/prompt", req))["prompt_id"]
print(f"Generating {DURATION}s of music. This runs in the background - watch the timer.\n")

out_files = []
while time.time() - t0 < 7200:
    h = json.load(urllib.request.urlopen(f"{S}/history/{pid}"))
    if pid in h:
        st = h[pid]["status"]
        el = time.time() - t0
        print(f"\n{st.get('status_str')} in {el:.0f}s ({el/60:.1f} min)")
        for m in st.get("messages", []):
            if m[0] == "execution_error":
                print("ERROR:", json.dumps(m[1], indent=1)[:1200])
        for o in h[pid].get("outputs", {}).values():
            for a in o.get("audio", []):
                src = os.path.join(COMFY, "output", a.get("subfolder", ""), a["filename"])
                dst = os.path.join("/kaggle/working", a["filename"])
                if os.path.exists(src):
                    import shutil; shutil.copy(src, dst); out_files.append(dst)
                    print("SAVED ->", dst)
        break
    print(f"  {time.time()-t0:6.0f}s elapsed", end="\r")
    time.sleep(10)
else:
    print("Timed out.")


## Step 6: listen

Hit play below.

To keep it: look at the **Output** panel on the right side of the screen and download the file from there.

In [ ]:
from IPython.display import Audio, display

if out_files:
    for f in out_files:
        print(f, f"{os.path.getsize(f)/1e6:.1f} MB")
        display(Audio(f))
else:
    print("No audio produced - check the error above, and the log:")
    print(open("/kaggle/temp/comfy.log", errors="ignore").read()[-3000:])


---
## A few things worth knowing

**Making another song is fast.** Change the words in YOUR SONG, then run just the last two
boxes. It does not have to fetch anything again, so you skip straight to the music.

**The description matters more than any setting.** "A sad song" gives you mush. "Slow country
ballad, 70 BPM, brushed drums, pedal steel, weary older male voice close to the mic" gives you
something you can use. Name the instruments. Name the mood. Say what the voice sounds like.

**If a song comes out wrong**, change SEED first before you change anything else. Often the
words were fine and you just got a bad take.

**If the session disconnects**, Kaggle stops after 9 hours or if you leave it idle too long.
Start it again and run from the top.

**One legal note:** if you ever sell anything made with this, the makers require the name
"MiniMax-Music3" shown somewhere in your product. That only kicks in for commercial use.
